In [76]:
!pip install -qU langchain langchain-community langchain-openai langchain-chroma chromadb pypdf openai sentence-transformers langchain-ollama langchain-classic


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [77]:
# Importações básicas
import os
import shutil
import requests

# loader de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

# Divisão de texto em blocos
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Banco vetorial (use apenas este)
from langchain_chroma import Chroma

# LLM (LM Studio / OpenAI-compatible)
from langchain_openai import ChatOpenAI

# Embeddings locais (sem OpenAI / sem LM Studio)
from sentence_transformers import SentenceTransformer

# Cadeia pronta de Perguntas e Respostas com RAG
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

from pathlib import Path


In [78]:

# Define a pasta onde estão os PDFs
pasta = Path("C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy")

# Busca automaticamente todos os arquivos .pdf da pasta
caminhos_bulas = list(pasta.glob("*.pdf"))
# Lista que armazenará todos os documentos carregados
documentos = []

# Percorre cada bula
for caminho in caminhos_bulas:

    # Cria o loader do PDF
    loader = PyPDFLoader(caminho)

    # Carrega o conteúdo do PDF
    docs = loader.load()

    # Adiciona o nome do medicamento como metadado
    for doc in docs:
        doc.metadata["medicamento"] = caminho.stem   

    # Adiciona os documentos à lista principal
    documentos.extend(docs)

# Exibe a quantidade total de páginas carregadas
len(documentos)


11

In [79]:
for doc in documentos:
    print(doc)
    print("-" * 50)

page_content='dipirona monoidratada 
 
Brainfarma Indústria Química e Farmacêutica S.A. 
 
 
Comprimido  
1g' metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy\\dipirona.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'medicamento': 'dipirona'}
--------------------------------------------------
page_content='dipirona monoidratada – comprimido - Bula para o paciente  1 
 
I - IDENTIFICAÇÃO DO MEDICAMENTO: 
 
dipirona monoidratada 
Medicamento genérico Lei nº 9.787, de 1999. 
 
APRESENTAÇÕES 
Comprimido. 
Embalagens contendo 10 ou 100 comprimidos.  
 
VIA DE ADMINISTRAÇÃO: ORAL 
 
USO ADULTO E PEDIÁTRICO ACIMA DE 15 ANOS. 
 
COMPOSIÇÃO 
Cada comprimido contém: 
dipirona monoidratada ...................................................................................................................... 1000mg 
excipientes q.s.

In [80]:

# Cria o objeto responsável pelo chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,      # Tamanho máximo de cada chunk
    chunk_overlap=150   # Sobreposição entre chunks
)

# Divide os documentos em chunks
chunks = text_splitter.split_documents(documentos)

# Quantidade total de chunks gerados
len(chunks)


28

In [81]:
chunks

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy\\dipirona.pdf', 'total_pages': 10, 'page': 0, 'page_label': '1', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada \n \nBrainfarma Indústria Química e Farmacêutica S.A. \n \n \nComprimido  \n1g'),
 Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy\\dipirona.pdf', 'total_pages': 10, 'page': 1, 'page_label': '2', 'medicamento': 'dipirona'}, page_content='dipirona monoidratada – comprimido - Bula para o paciente  1 \n \nI - IDENTIFICAÇÃO DO MEDICAMENTO: \n \ndipirona monoidratada \nMedicamento genérico Lei nº 9.787, de 1999. \n \nAPRESENTAÇÕES \nComprimido. \nEmbalagens contendo 10 ou 100 comp

In [82]:

# Percorre cada chunk para classificar semanticamente seu conteúdo
for chunk in chunks:

    # Normaliza o texto para facilitar as verificações
    texto = chunk.page_content.lower()

    # Identificação do medicamento
    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"

    # Indicações terapêuticas
    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"

    # Funcionamento do medicamento
    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"

    # Contraindicações
    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"

    # Advertências e precauções
    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"

    # Interações medicamentosas
    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"

    # Posologia e modo de uso
    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"

    # Reações adversas
    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"

    # Armazenamento
    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"

    # Superdosagem
    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"

    # Conteúdo geral / administrativo
    else:
        chunk.metadata["categoria"] = "geral"



In [83]:
import random

# Seleciona dois chunks aleatórios
chunks_aleatorios = random.sample(chunks, 2)

# Imprime os metadados e um trecho do conteúdo
for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(f"\n--- Chunk Aleatório {i} ---")
    print("Metadados:")
    print(chunk.metadata)
    print("\nConteúdo (início):")
    print(chunk.page_content[:300])



--- Chunk Aleatório 1 ---
Metadados:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy\\dipirona.pdf', 'total_pages': 10, 'page': 6, 'page_label': '7', 'medicamento': 'dipirona', 'categoria': 'como_funciona'}

Conteúdo (início):
dipirona monoidratada – comprimido - Bula para o paciente  6 
 
brancos - no sangue, em consequência de um distúrbio na medula óss ea) e pancitopenia (redução de 
glóbulos vermelhos, brancos e plaquetas), incluindo casos fatais, leucopenia (redução dos glóbulos 
brancos) e trombocitopenia (diminuiçã

--- Chunk Aleatório 2 ---
Metadados:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': 'C:\\Users\\Administrador\\OneDrive\\Documentos\\Profissional\\Alura\\2_Engenharia_IA\\Arq_RAG\\PDFs\\pharmacy\\dipirona.pdf', 'total_pages': 10, 'page': 9, 'page_label': '10', 'medicamento'

In [84]:
# Define a pasta onde o banco vetorial será salvo
persist_dir = str(Path("chroma_regras_farmaceuticas"))

# Modelo de embeddings local (rápido e leve)
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")  # 384 dims

texts = [d.page_content for d in chunks]
metadatas = [d.metadata for d in chunks]

# Gera embeddings localmente
vectors = embed_model.encode(texts, normalize_embeddings=True).tolist()

# Cria Chroma persistente e adiciona os dados
vectorstore = Chroma(
    collection_name="regras_farmaceuticas",
    persist_directory=persist_dir
)

vectorstore._collection.add(
    ids=[str(i) for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=vectors
)

print("Indexação concluída. Total docs:", len(texts))
print("Banco salvo em:", persist_dir)

Indexação concluída. Total docs: 28
Banco salvo em: chroma_regras_farmaceuticas


In [85]:

# Cria o retriever para busca semântica
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}  # Número de chunks retornados
)


In [86]:
docs_recuperados = retriever.invoke("Quais são as contraindicações da dipirona?")

for i, doc in enumerate(docs_recuperados, start=1):
    print(f"\nDocumento {i}")
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print(doc.page_content[:1000])
    print("-" * 80)


Documento 1
Medicamento: dipirona
Página: 3
A amamentação deve ser evitada durante e por até 48 horas após o uso d a dipirona monoidratada . A 
dipirona é eliminada no leite materno. 
 
Pacientes idosos:  deve-se considerar a possibilidade da s funções do fígado e dos rins estarem 
prejudicadas. 
Crianças: menores de 3 meses ou pesando menos de 5kg não devem ser tratadas com dipirona. 
A dipirona  monoidratada comprimido não é recomendada para menores de 15 anos. É recomendada 
supervisão médica quando se administra dipirona a crianças pequenas.
--------------------------------------------------------------------------------

Documento 2
Medicamento: dipirona
Página: 5
com risco à vida e, em alguns casos, serem fatais. Es tas reações podem ocorrer mesmo após  a dipirona  
monoidratada ter sido utilizada previamente em muitas ocasiões sem complicações. 
Estas reações medicamentosas podem desenvolver -se imediatamente após a administração d a dipirona ou 
horas mais tarde; contudo, a te

In [87]:
# LLM local via LM Studio (OpenAI-compatible server)
llm = ChatOpenAI(
    model="google/gemma-3-1b",  # ex.: qwen2.5-7b-instruct
    base_url="http://localhost:1235/v1",
    api_key="lm-studio",  # pode ser um valor dummy se a autenticação estiver desligada
    temperature=0
)

# Cadeia RAG
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [88]:
resp = llm.invoke("Responda apenas: LM Studio conectado.")
print(resp.content)

LM Studio conectado.


In [91]:
# Pergunta do usuário
pergunta = "Com base apenas no contexto, liste as contraindicações, restrições de uso e principais advertências da dipirona."

# Executa a consulta na cadeia RAG
resposta = qa_chain.invoke({"query": pergunta})

# Exibe a resposta gerada pelo modelo
print("RESPOSTA:")
print(resposta["result"])



RESPOSTA:
Aqui estão as informações sobre a dipirona, conforme o texto fornecido:

**Contraindicações:**

*   Síndrome da asma analgésica ou intolerância analgésica do tipo urticária-angioedema
*   Asma brônquica, particula rmente aqueles com rinossinusite poliposa (processo inflamatório no nariz e seios da face com formação de pólipos) concomitante
*   Urticária crônica
*   Intolerância ao álcool (reagindo até mesmo a pequenas quantidades de bebidas alcoólicas) apresentando sintomas como espirros, lacrimejamento e vermelhidão acentuada da face.
*   Intolerância a corantes ou conservantes (ex.: tartrazina e/ou benzoatos).

**Restrições de uso:**

*   Gravidez e amamentação: recomenda-se não usar durante os primeiros 3 meses da gravidez e o segundo trimestre da gravidez só deve ocorrer após cuidadosa avaliação do potencial risco/benefício pelo médico.
*   A dipirona monoidratada comprimido não é recomendada para menores de 15 anos. É recomendada supervisão médica quando se administra di

In [97]:
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(
        f"Fonte {i} | "
        f"Medicamento: {doc.metadata.get('medicamento', 'N/A')} | "
        f"Página: {doc.metadata.get('page', 'N/A')}"
    )

Fonte 1 | Medicamento: dipirona | Página: 5
Fonte 2 | Medicamento: dipirona | Página: 3
Fonte 3 | Medicamento: dipirona | Página: 4
Fonte 4 | Medicamento: dipirona | Página: 3
Fonte 5 | Medicamento: dipirona | Página: 3
Fonte 6 | Medicamento: dipirona | Página: 2


In [98]:
# Exibe os documentos/fontes usados na resposta
print("\nFONTES:")
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"\nFonte {i}:")
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")
    print(f"Trecho:\n{doc.page_content[:300]}")
    print("-" * 80)


FONTES:

Fonte 1:
Medicamento: dipirona
Página: 5
Trecho:
com risco à vida e, em alguns casos, serem fatais. Es tas reações podem ocorrer mesmo após  a dipirona  
monoidratada ter sido utilizada previamente em muitas ocasiões sem complicações. 
Estas reações medicamentosas podem desenvolver -se imediatamente após a administração d a dipirona ou 
horas mais
--------------------------------------------------------------------------------

Fonte 2:
Medicamento: dipirona
Página: 3
Trecho:
A amamentação deve ser evitada durante e por até 48 horas após o uso d a dipirona monoidratada . A 
dipirona é eliminada no leite materno. 
 
Pacientes idosos:  deve-se considerar a possibilidade da s funções do fígado e dos rins estarem 
prejudicadas. 
Crianças: menores de 3 meses ou pesando menos 
--------------------------------------------------------------------------------

Fonte 3:
Medicamento: dipirona
Página: 4
Trecho:
A dipirona monoidratada apresenta-se como comprimido oblongo, branco a leveme

In [99]:
# Pergunta do usuário
pergunta = """
Com base apenas no contexto recuperado, liste:
1. contraindicações
2. restrições de uso
3. advertências

Não invente informações.
Se alguma informação não estiver explícita no texto, diga "não identificado no contexto".
"""
# Executa a consulta na cadeia RAG
resposta = qa_chain.invoke({"query": pergunta})

# Exibe a resposta gerada pelo modelo
print("RESPOSTA:")
print(resposta["result"])



RESPOSTA:
Here's a breakdown of the information provided in the context, based on your request:

1.  **Contraindicações:**
    *   Síndrome da asma analgésica ou intolerância analgésica do tipo urticária-angioedema
    *   Asma brônquica, particulamente aqueles com rinossinusite poliposa (processo inflamatório no nariz e seios da face com formação de pólipos) concomitante.
    *   Urticária crônica
    *   Intolerância ao álcool (especialmente quando consumido em grandes quantidades).

2.  **Restrições de uso:**
    *   Não deve ser utilizado durante o tratamento em curto prazo.
    *   Não há experiência com o uso prolongado em pacientes com insuficiência nos rins ou no fígado.

3.  **Advertências:**
    *   Reações anafiláticas/anafilactóides (reação alérgica grave e imediata que pode levar à morte).
    *   Em algumas situações, reações medicamentosas podem desenvolver-se imediatamente após a administração da dipirona ou horas mais tarde; contudo, a tendência normal é que estes even